## Imports

In [1]:
import Pkg; Pkg.activate(@__DIR__); Pkg.instantiate();
# using Piccolo
using PiccoloQuantumObjects
using QuantumCollocation
using ForwardDiff
using LinearAlgebra
# using Plots
using SparseArrays
using Statistics
using CairoMakie
using Random
using NamedTrajectories

  Activating project at `~/Documents/research/pulses/project/notebooks/src`
┌ Warning: Circular dependency detected.
│ Precompilation will be skipped for dependencies in this cycle:
│  ┌ Piccolissimo
│  └─ QuantumCollocation
└ @ Base.Precompilation precompilation.jl:651
┌ Warning: Circular dependency detected.
│ Precompilation will be skipped for dependencies in this cycle:
│  ┌ Piccolissimo
│  └─ QuantumCollocation
└ @ Base.Precompilation precompilation.jl:651
┌ Warning: Replacing docs for `QuantumCollocation.ProblemTemplates.UnitaryUniversalProblem :: Union{}` in module `QuantumCollocation.ProblemTemplates`
└ @ Base.Docs docs/Docs.jl:243
ERROR: Method overwriting is not permitted during Module precompilation. Use `__precompile__(false)` to opt-out of precompilation.
┌ Warning: Replacing docs for `QuantumCollocation.ProblemTemplates.UnitaryUniversalProblem :: Union{}` in module `QuantumCollocation.ProblemTemplates`
└ @ Base.Docs docs/Docs.jl:243


In [2]:
# Problem parameters
T = 40
Δt = 0.8
U_goal = GATES.H
H_drive = [PAULIS.X, PAULIS.Y, PAULIS.Z]
piccolo_opts = PiccoloOptions(verbose=false)
pretty_print(X::AbstractMatrix) = Base.show(stdout, "text/plain", X);
sys = QuantumSystem(H_drive)
seeds = rand(1:1000, 25)
F=0.9999
num_iter = 6000
hess = false
hess_iter = 120
Qs = 10 .^ range(-4.0, 1.0, length=25)
a_bound = 1.0
dda_bound = 0.5
R=5e-3

0.005

In [3]:
∂ₑH = [PAULIS.Z]
H_drives = [PAULIS.X, PAULIS.Y, PAULIS.Z]
error_ops = [PAULIS.Z]

function var_obj(
    traj::NamedTrajectory, 
    H_drives::Vector{Matrix{ComplexF64}}, 
    H_errors::Vector{Matrix{ComplexF64}}
)
    Δt = traj.Δt[1]
    varsys = VariationalQuantumSystem(H_drives, H_errors)
    Ũ⃗, ∂Ũ⃗ = variational_unitary_rollout(traj, varsys)

    U = iso_vec_to_operator(Ũ⃗[:, end])
    # First error term
    ∂U = iso_vec_to_operator(∂Ũ⃗[1][:, end])

    d = size(U, 1)
    return abs(tr((U'*∂U)'*(U'*∂U))) / (T * Δt)^2 / d
end

# J_var = var_obj(var_prob.trajectory, H_drives, error_ops)

var_obj (generic function with 1 method)

In [4]:
function tog_obj(
    traj::NamedTrajectory, 
    H_drives::Vector{Matrix{ComplexF64}},
    H_error::Matrix{ComplexF64}
)
    T = traj.T
    Δt = get_timesteps(traj)

    sys = QuantumSystem(H_drives)
    U = iso_vec_to_operator.(eachcol(unitary_rollout(traj, sys)))
    
    # Toggle integral
    H_ti = sum(Δt[i] .* U[i]' * H_error * U[i] for i=1:T-1)

    d₁ = size(U[1], 1)
    Δt₁ = Δt[1]
    metric = norm(tr(H_ti'H_ti)) / (T * Δt₁)^2 / d₁
    return metric
end


tog_obj (generic function with 1 method)

In [6]:
function commutator(A::AbstractMatrix{<:Number}, B::AbstractMatrix{<:Number})
    return A*B - B*A
end

commutator (generic function with 1 method)

In [7]:
function pert_tog_obj(
    traj::NamedTrajectory, 
    H_drives::Vector{Matrix{ComplexF64}},
    H_error::Matrix{ComplexF64};
    order::Int=1,
    a_bound::Float64=a_bound
)
    T = traj.T
    Δt = get_timesteps(traj)

    sys = QuantumSystem(H_drives)
    U = iso_vec_to_operator.(eachcol(unitary_rollout(traj, sys)))

    # toggle integral
    H_ti = zeros(ComplexF64, size(U[1]))

    # note: U_1 = I, so U[:, k] = U_{k-1}.
    # you need to go to T-1, only
    for k in 1:T-1
        Hₖ = sum(traj.a[l, k] / a_bound * H for (l, H) in enumerate(H_drives))
        adjⁿH_E = H_error
        Eₖ_n = H_error * Δt[k]
        
        # get the different orders of the Hadamard lemma
        for n in 2:order
            coef_n = ComplexF64(im^(n-1) * a_bound^(n-1) * Δt[k]^n / factorial(big(n)))
            adjⁿH_E = commutator(Hₖ, adjⁿH_E)
            # Eₖ_n = push!(Eₖ_n, coef_n * adjⁿH_E)
            Eₖ_n += coef_n * adjⁿH_E
        end

        # nth order toggle integral up to k
        H_ti += U[k]' * Eₖ_n * U[k]
    end

    d₁ = size(U[1], 1)
    Δt₁ = Δt[1]
    metric = norm(tr(H_ti'H_ti)) / (T * Δt₁)^2 / d₁
    return metric
end

pert_tog_obj (generic function with 1 method)

In [9]:
# Directories where data is saved
var_dir = "artifacts/var_gap_data_export"
tog_dir = "artifacts/tog_gap_data_export"

# Load the first seed file to get metadata
var_first = load(joinpath(var_dir, "var_probs_seed_idx_1.jld2"))
tog_first = load(joinpath(tog_dir, "htog_probs_seed_idx_1.jld2"))

# Extract common parameters
Qs = var_first["Qs"]

n_seeds = length(readdir(var_dir)) - 1  # Subtract 1 for plot.png

# Initialize arrays to store loaded data
var_probs = Matrix{Any}(undef, n_seeds, length(Qs))
tog_probs = Matrix{Any}(undef, n_seeds, length(Qs))

# Load all var_probs data
for i in 1:n_seeds
    data = load(joinpath(var_dir, "var_probs_seed_idx_$(i).jld2"))
    var_probs[i, :] = data["var_probs"]
end

# Load all htog_probs data
for i in 1:n_seeds
    data = load(joinpath(tog_dir, "htog_probs_seed_idx_$(i).jld2"))
    tog_probs[i, :] = data["htog_probs"]
end

println("Loaded data for $(n_seeds) seeds across $(length(Qs)) Q values")

Loaded data for 25 seeds across 25 Q values


In [21]:
using JLD2, FileIO

# Create the directory
artifacts_dir = "artifacts/def_gap_data_export_new"
mkpath(artifacts_dir)

# Save htog_probs data for each seed in separate files
for (i, seed) in enumerate(seeds)
    save(joinpath(artifacts_dir, "probs_seed_idx_$(i).jld2"), Dict(
        "probs" => probs[i, :],
        "seed" => seed,
        "Qs" => Qs,
        "F" => F,
        "num_iter" => num_iter,
        "hess_iter" => hess_iter,
        "a_bound" => a_bound,
        "dda_bound" => dda_bound,
        "R" => R
    ))
end

# Save the figure
save(joinpath(artifacts_dir, "plot.png"), fig)

println("Data saved to $artifacts_dir")
println("Created $(length(seeds)) separate files, one for each seed")

UndefVarError: UndefVarError: `probs` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name may be made accessible by importing Distributions in the current active module Main

In [10]:
var_prob = var_probs[5,21]#[16, 18]
tog_prob = tog_probs[5,21]#[15, 18]
prob = probs[5,21]
x1 = 0.001
x2 = 0.01
steps = 10
step_size = (x2 - x1) / steps
epsilons = x1:step_size:x2
logrange(x1, x2, n) = 10 .^ range(log10(x1), log10(x2), length=n)
epsilons_zoom = logrange(x1, x2, steps)


UndefVarError: UndefVarError: `probs` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: a global variable of this name may be made accessible by importing Distributions in the current active module Main

In [11]:

colors = Makie.wong_colors()
ket_0 = [1.0,0.0]
rho_0 = ket_0 * ket_0'
var_traj = var_prob.trajectory
T = var_traj.T
tog_traj = tog_prob.trajectory
traj = prob.trajectory
# Original trajectories
expect_val_x = [real(tr(PAULIS.X * iso_vec_to_operator(var_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(var_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_y = [real(tr(PAULIS.Y * iso_vec_to_operator(var_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(var_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_z = [real(tr(PAULIS.Z * iso_vec_to_operator(var_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(var_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_x_1 = [real(tr(PAULIS.X * iso_vec_to_operator(tog_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(tog_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_y_1 = [real(tr(PAULIS.Y * iso_vec_to_operator(tog_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(tog_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_z_1 = [real(tr(PAULIS.Z * iso_vec_to_operator(tog_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(tog_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_x_2 = [real(tr(PAULIS.X * iso_vec_to_operator(traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_y_2 = [real(tr(PAULIS.Y * iso_vec_to_operator(traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_z_2 = [real(tr(PAULIS.Z * iso_vec_to_operator(traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(traj.Ũ⃗[:, t])')) for t in 1:T]

# Perturbed systems
H_drive = [PAULIS.X, PAULIS.Y, PAULIS.Z]

# Store trajectories for perturbed systems
var_perturbed_trajectories = []
tog_perturbed_trajectories = []
perturbed_trajectories = []
for eps in epsilons
    # Create perturbed system
    perturbed_sys = QuantumSystem(eps*PAULIS.Z, H_drive)
    Ũ⃗ = unitary_rollout(traj, perturbed_sys)
    Ũ⃗_var = unitary_rollout(var_traj, perturbed_sys)
    Ũ⃗_tog = unitary_rollout(tog_traj, perturbed_sys)
    # You'll need to solve for the trajectory here using your problem setup
    # This assumes you have a similar setup for getting the trajectory
    # Replace this with your actual problem solving code
    # perturbed_prob = var_prob  # This is a placeholder - you need to solve for the perturbed system
    # perturbed_traj = perturbed_prob.trajectory
    
    # Calculate expectation values for perturbed trajectory
    var_exp_x = [real(tr(PAULIS.X * iso_vec_to_operator(Ũ⃗_var[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_var[:, t])')) for t in 1:T]
    var_exp_y = [real(tr(PAULIS.Y * iso_vec_to_operator(Ũ⃗_var[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_var[:, t])')) for t in 1:T]
    var_exp_z = [real(tr(PAULIS.Z * iso_vec_to_operator(Ũ⃗_var[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_var[:, t])')) for t in 1:T]
    tog_exp_x = [real(tr(PAULIS.X * iso_vec_to_operator(Ũ⃗_tog[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_tog[:, t])')) for t in 1:T]
    tog_exp_y = [real(tr(PAULIS.Y * iso_vec_to_operator(Ũ⃗_tog[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_tog[:, t])')) for t in 1:T]
    tog_exp_z = [real(tr(PAULIS.Z * iso_vec_to_operator(Ũ⃗_tog[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_tog[:, t])')) for t in 1:T]
    exp_x = [real(tr(PAULIS.X * iso_vec_to_operator(Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗[:, t])')) for t in 1:T]
    exp_y = [real(tr(PAULIS.Y * iso_vec_to_operator(Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗[:, t])')) for t in 1:T]
    exp_z = [real(tr(PAULIS.Z * iso_vec_to_operator(Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗[:, t])')) for t in 1:T]

    push!(var_perturbed_trajectories, (var_exp_x, var_exp_y, var_exp_z, eps))
    push!(tog_perturbed_trajectories, (tog_exp_x, tog_exp_y, tog_exp_z, eps))
    push!(perturbed_trajectories, (exp_x, exp_y, exp_z, eps))

end

# Plotting
using CairoMakie
using GeometryBasics

f  = CairoMakie.Figure(resolution = (1600, 1200))
ax = CairoMakie.Axis3(f[1, 1];
    aspect = :equal
)

palette = to_colormap(:tab10)
styles  = (:solid, :dash, :dot, :dashdot)

origins = [Point3f(0,0,0), Point3f(0,0,0), Point3f(0,0,0)]
dirs    = [Vec3f(1.0,0,0), Vec3f(0,1.0,0), Vec3f(0,0,1.0)]

CairoMakie.arrows!(ax, origins, dirs;
    color = [:red, :green, :blue],
    arrowsize = 0.05,
    linewidth = 0.01
)

CairoMakie.text!(ax, "x", position = Point3f(1.2, 0, 0), align = (:left, :center),  color = :red,   fontsize = 28)
CairoMakie.text!(ax, "y", position = Point3f(0, 1.2, 0), align = (:center, :bottom), color = :green, fontsize = 28)
CairoMakie.text!(ax, "z", position = Point3f(0, 0, 1.2), align = (:center, :bottom), color = :blue,  fontsize = 28)

# Plot original trajectories
CairoMakie.lines!(ax, real.(expect_val_x), real.(expect_val_y), real.(expect_val_z);
    color     = colors[5],
    linestyle = styles[1],
    label     = "|0⟩ to |+⟩, Robust",
    linewidth = 1.5
)

# CairoMakie.lines!(ax, real.(expect_val_x_1), real.(expect_val_y_1), real.(expect_val_z_1);
#     color     = colors[3],
#     linestyle = styles[1],
#     label     = "|0⟩ to |+⟩, toggle",
#     linewidth = 1.5
# )

CairoMakie.lines!(ax, real.(expect_val_x_2), real.(expect_val_y_2), real.(expect_val_z_2);
    color     = :black,
    linestyle = styles[1],
    label     = "|0⟩ to |+⟩, Default",
    linewidth = 1.5
)


# Plot perturbed trajectories with fading colors
for (i, (var_exp_x, var_exp_y, var_exp_z, eps)) in enumerate(var_perturbed_trajectories)
    # Calculate alpha (transparency) based on epsilon value
    # Higher epsilon = more transparent (fainter)
    alpha = 1.0 - (1.1*i-1) / (length(epsilons) + 1)  # Fade from 1.0 to ~0.25
    
    # Use a different color from the palette and adjust with RGBA
    color_with_alpha = (colors[5], alpha)
    
    CairoMakie.lines!(ax, real.(var_exp_x), real.(var_exp_y), real.(var_exp_z);
        color     = color_with_alpha,
        linestyle = styles[1],
        # label     = "Perturbed ε=$(eps)",
        linewidth = 2.0 - 0.15*i  # Also make lines thinner as epsilon increases
    )
    
    # Add endpoint markers for perturbed trajectories
    CairoMakie.scatter!(ax, [real(var_exp_x[end])], [real(var_exp_y[end])], [real(var_exp_z[end])];
        color = color_with_alpha, 
        markersize = 6.0 - 0.15*i,  # Smaller markers for higher epsilon
        marker = :diamond
    )
end

# # Plot perturbed trajectories with fading colors
# for (i, (tog_exp_x, tog_exp_y, tog_exp_z, eps)) in enumerate(tog_perturbed_trajectories)
#     # Calculate alpha (transparency) based on epsilon value
#     # Higher epsilon = more transparent (fainter)
#     alpha = 1.0 - (i-1) / (length(epsilons) + 1)  # Fade from 1.0 to ~0.25
    
#     # Use a different color from the palette and adjust with RGBA
#     color_with_alpha = (colors[3], alpha)
    
#     CairoMakie.lines!(ax, real.(tog_exp_x), real.(tog_exp_y), real.(tog_exp_z);
#         color     = color_with_alpha,
#         linestyle = styles[1],
#         # label     = "Perturbed ε=$(eps)",
#         linewidth = 2.0 - 0.3*i  # Also make lines thinner as epsilon increases
#     )
    
#     # Add endpoint markers for perturbed trajectories
#     CairoMakie.scatter!(ax, [real(tog_exp_x[end])], [real(tog_exp_y[end])], [real(tog_exp_z[end])];
#         color = color_with_alpha, 
#         markersize = 4.0 - 0.3*i,  # Smaller markers for higher epsilon
#         marker = :diamond
#     )
# end

# Plot perturbed trajectories with fading colors
for (i, (exp_x, exp_y, exp_z, eps)) in enumerate(perturbed_trajectories)
    # Calculate alpha (transparency) based on epsilon value
    # Higher epsilon = more transparent (fainter)
    alpha = 0.9 - (1.15*i-1) / (length(epsilons) + 1)  # Fade from 1.0 to ~0.25
    
    # Use a different color from the palette and adjust with RGBA
    color_with_alpha = (:black, alpha)
    
    CairoMakie.lines!(ax, real.(exp_x), real.(exp_y), real.(exp_z);
        color     = color_with_alpha,
        linestyle = styles[1],
        # label     = "Perturbed ε=$(eps)",
        linewidth = 2.0 - 0.15*i  # Also make lines thinner as epsilon increases
    )
    
    # Add endpoint markers for perturbed trajectories
    CairoMakie.scatter!(ax, [real(exp_x[end])], [real(exp_y[end])], [real(exp_z[end])];
        color = color_with_alpha, 
        markersize = 6.0 - 0.15*i,  # Smaller markers for higher epsilon
        marker = :diamond
    )
end


# Original endpoint markers
CairoMakie.scatter!(ax, [real(expect_val_x[end])], [real(expect_val_y[end])], [real(expect_val_z[end])];
    color = colors[5], markersize = 20, marker = :xcross)

# CairoMakie.scatter!(ax, [real(expect_val_x_1[end])], [real(expect_val_y_1[end])], [real(expect_val_z_1[end])];
#     color = colors[3], markersize = 15, marker = :circle)

CairoMakie.scatter!(ax, [real(expect_val_x_2[end])], [real(expect_val_y_2[end])], [real(expect_val_z_2[end])];
    color = :black, markersize = 15, marker = :circle)

CairoMakie.mesh!(ax, Sphere(Point3f(0,0,0), 1f0);
    color = (0.2, 0.6, 1.0, 0.05),
    transparency = true,
    shading = true
)

# CairoMakie.xlims!(ax, -1.3, 1.3)
# CairoMakie.ylims!(ax, -1.3, 1.3)
# CairoMakie.zlims!(ax, -1.3, 1.3)

ax.azimuth[]   =  π/20
ax.elevation[] =  π/10
CairoMakie.hidespines!(ax)
CairoMakie.axislegend(ax; 
    position = :rt,
    labelsize = 36,        # Default is usually ~12-16. Increase this for text.
    patchsize = (100, 80),  # (width, height). Makes the line sample longer.
    linewidth = 3,         # Makes the line inside the legend thicker.
    padding = (10, 10, 10, 10) # Adds space inside the legend box.
)

CairoMakie.hidexdecorations!(ax, grid = true)
CairoMakie.hideydecorations!(ax, grid = true)
CairoMakie.hidezdecorations!(ax, grid = true)

f



UndefVarError: UndefVarError: `prob` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [12]:
save("./artifacts/plots/bloch_full.svg", f, px_per_unit = 3)

UndefVarError: UndefVarError: `f` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [13]:

colors = Makie.wong_colors()
ket_0 = [1.0,0.0]
rho_0 = ket_0 * ket_0'
var_traj = var_prob.trajectory
T = var_traj.T
tog_traj = tog_prob.trajectory
traj = prob.trajectory
# Original trajectories
expect_val_x = [real(tr(PAULIS.X * iso_vec_to_operator(var_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(var_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_y = [real(tr(PAULIS.Y * iso_vec_to_operator(var_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(var_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_z = [real(tr(PAULIS.Z * iso_vec_to_operator(var_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(var_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_x_1 = [real(tr(PAULIS.X * iso_vec_to_operator(tog_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(tog_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_y_1 = [real(tr(PAULIS.Y * iso_vec_to_operator(tog_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(tog_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_z_1 = [real(tr(PAULIS.Z * iso_vec_to_operator(tog_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(tog_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_x_2 = [real(tr(PAULIS.X * iso_vec_to_operator(traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_y_2 = [real(tr(PAULIS.Y * iso_vec_to_operator(traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_z_2 = [real(tr(PAULIS.Z * iso_vec_to_operator(traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(traj.Ũ⃗[:, t])')) for t in 1:T]

# Perturbed systems
H_drive = [PAULIS.X, PAULIS.Y, PAULIS.Z]

# Store trajectories for perturbed systems
var_perturbed_trajectories = []
tog_perturbed_trajectories = []
perturbed_trajectories = []
for eps in epsilons
    # Create perturbed system
    perturbed_sys = QuantumSystem(eps*PAULIS.Z, H_drive)
    Ũ⃗ = unitary_rollout(traj, perturbed_sys)
    Ũ⃗_var = unitary_rollout(var_traj, perturbed_sys)
    Ũ⃗_tog = unitary_rollout(tog_traj, perturbed_sys)
    # You'll need to solve for the trajectory here using your problem setup
    # This assumes you have a similar setup for getting the trajectory
    # Replace this with your actual problem solving code
    # perturbed_prob = var_prob  # This is a placeholder - you need to solve for the perturbed system
    # perturbed_traj = perturbed_prob.trajectory
    
    # Calculate expectation values for perturbed trajectory
    var_exp_x = [real(tr(PAULIS.X * iso_vec_to_operator(Ũ⃗_var[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_var[:, t])')) for t in 1:T]
    var_exp_y = [real(tr(PAULIS.Y * iso_vec_to_operator(Ũ⃗_var[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_var[:, t])')) for t in 1:T]
    var_exp_z = [real(tr(PAULIS.Z * iso_vec_to_operator(Ũ⃗_var[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_var[:, t])')) for t in 1:T]
    tog_exp_x = [real(tr(PAULIS.X * iso_vec_to_operator(Ũ⃗_tog[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_tog[:, t])')) for t in 1:T]
    tog_exp_y = [real(tr(PAULIS.Y * iso_vec_to_operator(Ũ⃗_tog[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_tog[:, t])')) for t in 1:T]
    tog_exp_z = [real(tr(PAULIS.Z * iso_vec_to_operator(Ũ⃗_tog[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_tog[:, t])')) for t in 1:T]
    exp_x = [real(tr(PAULIS.X * iso_vec_to_operator(Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗[:, t])')) for t in 1:T]
    exp_y = [real(tr(PAULIS.Y * iso_vec_to_operator(Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗[:, t])')) for t in 1:T]
    exp_z = [real(tr(PAULIS.Z * iso_vec_to_operator(Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗[:, t])')) for t in 1:T]

    push!(var_perturbed_trajectories, (var_exp_x, var_exp_y, var_exp_z, eps))
    push!(tog_perturbed_trajectories, (tog_exp_x, tog_exp_y, tog_exp_z, eps))
    push!(perturbed_trajectories, (exp_x, exp_y, exp_z, eps))

end

# Plotting
using CairoMakie
using GeometryBasics

f  = CairoMakie.Figure(resolution = (1600, 1200))
ax = CairoMakie.Axis3(f[1, 1];
    aspect = :equal
)

palette = to_colormap(:tab10)
styles  = (:solid, :dash, :dot, :dashdot)

origins = [Point3f(0,0,0), Point3f(0,0,0), Point3f(0,0,0)]
dirs    = [Vec3f(1.0,0,0), Vec3f(0,1.0,0), Vec3f(0,0,1.0)]

CairoMakie.arrows!(ax, origins, dirs;
    color = [:red, :green, :blue],
    arrowsize = 0.05,
    linewidth = 0.01
)

CairoMakie.text!(ax, "x", position = Point3f(1.2, 0, 0), align = (:left, :center),  color = :red,   fontsize = 28)
CairoMakie.text!(ax, "y", position = Point3f(0, 1.2, 0), align = (:center, :bottom), color = :green, fontsize = 28)
CairoMakie.text!(ax, "z", position = Point3f(0, 0, 1.2), align = (:center, :bottom), color = :blue,  fontsize = 28)

# Plot original trajectories
CairoMakie.lines!(ax, real.(expect_val_x), real.(expect_val_y), real.(expect_val_z);
    color     = colors[5],
    linestyle = styles[1],
    label     = "|0⟩ to |+⟩, Robust",
    linewidth = 1.5
)

# CairoMakie.lines!(ax, real.(expect_val_x_1), real.(expect_val_y_1), real.(expect_val_z_1);
#     color     = colors[3],
#     linestyle = styles[1],
#     label     = "|0⟩ to |+⟩, toggle",
#     linewidth = 1.5
# )

# CairoMakie.lines!(ax, real.(expect_val_x_2), real.(expect_val_y_2), real.(expect_val_z_2);
#     color     = :black,
#     linestyle = styles[1],
#     label     = "|0⟩ to |+⟩, Default",
#     linewidth = 1.5
# )


# Plot perturbed trajectories with fading colors
for (i, (var_exp_x, var_exp_y, var_exp_z, eps)) in enumerate(var_perturbed_trajectories)
    # Calculate alpha (transparency) based on epsilon value
    # Higher epsilon = more transparent (fainter)
    alpha = 1.0 - (1.1*i-1) / (length(epsilons) + 1)  # Fade from 1.0 to ~0.25
    
    # Use a different color from the palette and adjust with RGBA
    color_with_alpha = (colors[5], alpha)
    
    CairoMakie.lines!(ax, real.(var_exp_x), real.(var_exp_y), real.(var_exp_z);
        color     = color_with_alpha,
        linestyle = styles[1],
        # label     = "Perturbed ε=$(eps)",
        linewidth = 2.0 - 0.15*i  # Also make lines thinner as epsilon increases
    )
    
    # Add endpoint markers for perturbed trajectories
    CairoMakie.scatter!(ax, [real(var_exp_x[end])], [real(var_exp_y[end])], [real(var_exp_z[end])];
        color = color_with_alpha, 
        markersize = 6.0 - 0.15*i,  # Smaller markers for higher epsilon
        marker = :diamond
    )
end

# # Plot perturbed trajectories with fading colors
# for (i, (tog_exp_x, tog_exp_y, tog_exp_z, eps)) in enumerate(tog_perturbed_trajectories)
#     # Calculate alpha (transparency) based on epsilon value
#     # Higher epsilon = more transparent (fainter)
#     alpha = 1.0 - (i-1) / (length(epsilons) + 1)  # Fade from 1.0 to ~0.25
    
#     # Use a different color from the palette and adjust with RGBA
#     color_with_alpha = (colors[3], alpha)
    
#     CairoMakie.lines!(ax, real.(tog_exp_x), real.(tog_exp_y), real.(tog_exp_z);
#         color     = color_with_alpha,
#         linestyle = styles[1],
#         # label     = "Perturbed ε=$(eps)",
#         linewidth = 2.0 - 0.3*i  # Also make lines thinner as epsilon increases
#     )
    
#     # Add endpoint markers for perturbed trajectories
#     CairoMakie.scatter!(ax, [real(tog_exp_x[end])], [real(tog_exp_y[end])], [real(tog_exp_z[end])];
#         color = color_with_alpha, 
#         markersize = 4.0 - 0.3*i,  # Smaller markers for higher epsilon
#         marker = :diamond
#     )
# end

# # Plot perturbed trajectories with fading colors
# for (i, (exp_x, exp_y, exp_z, eps)) in enumerate(perturbed_trajectories)
#     # Calculate alpha (transparency) based on epsilon value
#     # Higher epsilon = more transparent (fainter)
#     alpha = 0.9 - (1.15*i-1) / (length(epsilons) + 1)  # Fade from 1.0 to ~0.25
    
#     # Use a different color from the palette and adjust with RGBA
#     color_with_alpha = (:black, alpha)
    
#     CairoMakie.lines!(ax, real.(exp_x), real.(exp_y), real.(exp_z);
#         color     = color_with_alpha,
#         linestyle = styles[1],
#         # label     = "Perturbed ε=$(eps)",
#         linewidth = 2.0 - 0.15*i  # Also make lines thinner as epsilon increases
#     )
    
#     # Add endpoint markers for perturbed trajectories
#     CairoMakie.scatter!(ax, [real(exp_x[end])], [real(exp_y[end])], [real(exp_z[end])];
#         color = color_with_alpha, 
#         markersize = 6.0 - 0.15*i,  # Smaller markers for higher epsilon
#         marker = :diamond
#     )
# end


# Original endpoint markers
CairoMakie.scatter!(ax, [real(expect_val_x[end])], [real(expect_val_y[end])], [real(expect_val_z[end])];
    color = colors[5], markersize = 20, marker = :xcross)

# CairoMakie.scatter!(ax, [real(expect_val_x_1[end])], [real(expect_val_y_1[end])], [real(expect_val_z_1[end])];
#     color = colors[3], markersize = 15, marker = :circle)

# CairoMakie.scatter!(ax, [real(expect_val_x_2[end])], [real(expect_val_y_2[end])], [real(expect_val_z_2[end])];
#     color = :black, markersize = 15, marker = :circle)

CairoMakie.mesh!(ax, Sphere(Point3f(0,0,0), 1f0);
    color = (0.2, 0.6, 1.0, 0.05),
    transparency = true,
    shading = true
)

# CairoMakie.xlims!(ax, -1.3, 1.3)
# CairoMakie.ylims!(ax, -1.3, 1.3)
# CairoMakie.zlims!(ax, -1.3, 1.3)

ax.azimuth[]   =  π/20
ax.elevation[] =  π/10
CairoMakie.hidespines!(ax)
CairoMakie.axislegend(ax; 
    position = :rt,
    labelsize = 36,        # Default is usually ~12-16. Increase this for text.
    patchsize = (100, 80),  # (width, height). Makes the line sample longer.
    linewidth = 3,         # Makes the line inside the legend thicker.
    padding = (10, 10, 10, 10) # Adds space inside the legend box.
)

CairoMakie.hidexdecorations!(ax, grid = true)
CairoMakie.hideydecorations!(ax, grid = true)
CairoMakie.hidezdecorations!(ax, grid = true)

f



UndefVarError: UndefVarError: `prob` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [14]:
save("./artifacts/plots/bloch_rob.svg", f, px_per_unit = 2)

UndefVarError: UndefVarError: `f` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [15]:
colors = Makie.wong_colors()
ket_0 = [1.0,0.0]
rho_0 = ket_0 * ket_0'
var_traj = var_prob.trajectory
T = var_traj.T
tog_traj = tog_prob.trajectory
traj = prob.trajectory

# Original trajectories
expect_val_x = [real(tr(PAULIS.X * iso_vec_to_operator(var_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(var_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_y = [real(tr(PAULIS.Y * iso_vec_to_operator(var_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(var_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_z = [real(tr(PAULIS.Z * iso_vec_to_operator(var_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(var_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_x_1 = [real(tr(PAULIS.X * iso_vec_to_operator(tog_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(tog_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_y_1 = [real(tr(PAULIS.Y * iso_vec_to_operator(tog_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(tog_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_z_1 = [real(tr(PAULIS.Z * iso_vec_to_operator(tog_traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(tog_traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_x_2 = [real(tr(PAULIS.X * iso_vec_to_operator(traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_y_2 = [real(tr(PAULIS.Y * iso_vec_to_operator(traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(traj.Ũ⃗[:, t])')) for t in 1:T]
expect_val_z_2 = [real(tr(PAULIS.Z * iso_vec_to_operator(traj.Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(traj.Ũ⃗[:, t])')) for t in 1:T]

# Perturbed systems
H_drive = [PAULIS.X, PAULIS.Y, PAULIS.Z]

# Store trajectories for perturbed systems
var_perturbed_trajectories = []
tog_perturbed_trajectories = []
perturbed_trajectories = []
for eps in epsilons
    # Create perturbed system
    perturbed_sys = QuantumSystem(eps*PAULIS.Z, H_drive)
    Ũ⃗ = unitary_rollout(traj, perturbed_sys)
    Ũ⃗_var = unitary_rollout(var_traj, perturbed_sys)
    Ũ⃗_tog = unitary_rollout(tog_traj, perturbed_sys)
    # You'll need to solve for the trajectory here using your problem setup
    # This assumes you have a similar setup for getting the trajectory
    # Replace this with your actual problem solving code
    # perturbed_prob = var_prob  # This is a placeholder - you need to solve for the perturbed system
    # perturbed_traj = perturbed_prob.trajectory
    
    # Calculate expectation values for perturbed trajectory
    var_exp_x = [real(tr(PAULIS.X * iso_vec_to_operator(Ũ⃗_var[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_var[:, t])')) for t in 1:T]
    var_exp_y = [real(tr(PAULIS.Y * iso_vec_to_operator(Ũ⃗_var[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_var[:, t])')) for t in 1:T]
    var_exp_z = [real(tr(PAULIS.Z * iso_vec_to_operator(Ũ⃗_var[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_var[:, t])')) for t in 1:T]
    tog_exp_x = [real(tr(PAULIS.X * iso_vec_to_operator(Ũ⃗_tog[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_tog[:, t])')) for t in 1:T]
    tog_exp_y = [real(tr(PAULIS.Y * iso_vec_to_operator(Ũ⃗_tog[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_tog[:, t])')) for t in 1:T]
    tog_exp_z = [real(tr(PAULIS.Z * iso_vec_to_operator(Ũ⃗_tog[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗_tog[:, t])')) for t in 1:T]
    exp_x = [real(tr(PAULIS.X * iso_vec_to_operator(Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗[:, t])')) for t in 1:T]
    exp_y = [real(tr(PAULIS.Y * iso_vec_to_operator(Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗[:, t])')) for t in 1:T]
    exp_z = [real(tr(PAULIS.Z * iso_vec_to_operator(Ũ⃗[:, t]) * rho_0 * iso_vec_to_operator(Ũ⃗[:, t])')) for t in 1:T]

    push!(var_perturbed_trajectories, (var_exp_x, var_exp_y, var_exp_z, eps))
    push!(tog_perturbed_trajectories, (tog_exp_x, tog_exp_y, tog_exp_z, eps))
    push!(perturbed_trajectories, (exp_x, exp_y, exp_z, eps))

end

# Plotting
using CairoMakie
using GeometryBasics

f  = CairoMakie.Figure(resolution = (1600, 1200))
ax = CairoMakie.Axis3(f[1, 1];
    aspect = :equal
)

palette = to_colormap(:tab10)
styles  = (:solid, :dash, :dot, :dashdot)

origins = [Point3f(0,0,0), Point3f(0,0,0), Point3f(0,0,0)]
dirs    = [Vec3f(1.0,0,0), Vec3f(0,1.0,0), Vec3f(0,0,1.0)]

CairoMakie.arrows!(ax, origins, dirs;
    color = [:red, :green, :blue],
    arrowsize = 0.05,
    linewidth = 0.01
)

CairoMakie.text!(ax, "x", position = Point3f(1.2, 0, 0), align = (:left, :center),  color = :red,   fontsize = 28)
CairoMakie.text!(ax, "y", position = Point3f(0, 1.2, 0), align = (:center, :bottom), color = :green, fontsize = 28)
CairoMakie.text!(ax, "z", position = Point3f(0, 0, 1.2), align = (:center, :bottom), color = :blue,  fontsize = 28)

# Plot original trajectories
# CairoMakie.lines!(ax, real.(expect_val_x), real.(expect_val_y), real.(expect_val_z);
#     color     = colors[5],
#     linestyle = styles[1],
#     label     = "|0⟩ to |+⟩, Robust",
#     linewidth = 1.5
# )

# CairoMakie.lines!(ax, real.(expect_val_x_1), real.(expect_val_y_1), real.(expect_val_z_1);
#     color     = colors[3],
#     linestyle = styles[1],
#     label     = "|0⟩ to |+⟩, toggle",
#     linewidth = 1.5
# )

CairoMakie.lines!(ax, real.(expect_val_x_2), real.(expect_val_y_2), real.(expect_val_z_2);
    color     = :black,
    linestyle = styles[1],
    label     = "|0⟩ to |+⟩, Default",
    linewidth = 1.5
)


# Plot perturbed trajectories with fading colors
# for (i, (var_exp_x, var_exp_y, var_exp_z, eps)) in enumerate(var_perturbed_trajectories)
#     # Calculate alpha (transparency) based on epsilon value
#     # Higher epsilon = more transparent (fainter)
#     alpha = 1.0 - (1.1*i-1) / (length(epsilons) + 1)  # Fade from 1.0 to ~0.25
    
#     # Use a different color from the palette and adjust with RGBA
#     color_with_alpha = (colors[5], alpha)
    
#     CairoMakie.lines!(ax, real.(var_exp_x), real.(var_exp_y), real.(var_exp_z);
#         color     = color_with_alpha,
#         linestyle = styles[1],
#         # label     = "Perturbed ε=$(eps)",
#         linewidth = 2.0 - 0.15*i  # Also make lines thinner as epsilon increases
#     )
    
#     # Add endpoint markers for perturbed trajectories
#     CairoMakie.scatter!(ax, [real(var_exp_x[end])], [real(var_exp_y[end])], [real(var_exp_z[end])];
#         color = color_with_alpha, 
#         markersize = 6.0 - 0.15*i,  # Smaller markers for higher epsilon
#         marker = :diamond
#     )
# end

# # Plot perturbed trajectories with fading colors
# for (i, (tog_exp_x, tog_exp_y, tog_exp_z, eps)) in enumerate(tog_perturbed_trajectories)
#     # Calculate alpha (transparency) based on epsilon value
#     # Higher epsilon = more transparent (fainter)
#     alpha = 1.0 - (i-1) / (length(epsilons) + 1)  # Fade from 1.0 to ~0.25
    
#     # Use a different color from the palette and adjust with RGBA
#     color_with_alpha = (colors[3], alpha)
    
#     CairoMakie.lines!(ax, real.(tog_exp_x), real.(tog_exp_y), real.(tog_exp_z);
#         color     = color_with_alpha,
#         linestyle = styles[1],
#         # label     = "Perturbed ε=$(eps)",
#         linewidth = 2.0 - 0.3*i  # Also make lines thinner as epsilon increases
#     )
    
#     # Add endpoint markers for perturbed trajectories
#     CairoMakie.scatter!(ax, [real(tog_exp_x[end])], [real(tog_exp_y[end])], [real(tog_exp_z[end])];
#         color = color_with_alpha, 
#         markersize = 4.0 - 0.3*i,  # Smaller markers for higher epsilon
#         marker = :diamond
#     )
# end

# Plot perturbed trajectories with fading colors
for (i, (exp_x, exp_y, exp_z, eps)) in enumerate(perturbed_trajectories)
    # Calculate alpha (transparency) based on epsilon value
    # Higher epsilon = more transparent (fainter)
    alpha = 0.9 - (1.15*i-1) / (length(epsilons) + 1)  # Fade from 1.0 to ~0.25
    
    # Use a different color from the palette and adjust with RGBA
    color_with_alpha = (:black, alpha)
    
    CairoMakie.lines!(ax, real.(exp_x), real.(exp_y), real.(exp_z);
        color     = color_with_alpha,
        linestyle = styles[1],
        # label     = "Perturbed ε=$(eps)",
        linewidth = 2.0 - 0.15*i  # Also make lines thinner as epsilon increases
    )
    
    # Add endpoint markers for perturbed trajectories
    CairoMakie.scatter!(ax, [real(exp_x[end])], [real(exp_y[end])], [real(exp_z[end])];
        color = color_with_alpha, 
        markersize = 6.0 - 0.15*i,  # Smaller markers for higher epsilon
        marker = :diamond
    )
end


# # Original endpoint markers
# CairoMakie.scatter!(ax, [real(expect_val_x[end])], [real(expect_val_y[end])], [real(expect_val_z[end])];
#     color = colors[5], markersize = 20, marker = :xcross)

# CairoMakie.scatter!(ax, [real(expect_val_x_1[end])], [real(expect_val_y_1[end])], [real(expect_val_z_1[end])];
#     color = colors[3], markersize = 15, marker = :circle)

CairoMakie.scatter!(ax, [real(expect_val_x_2[end])], [real(expect_val_y_2[end])], [real(expect_val_z_2[end])];
    color = :black, markersize = 15, marker = :circle)

CairoMakie.mesh!(ax, Sphere(Point3f(0,0,0), 1f0);
    color = (0.2, 0.6, 1.0, 0.05),
    transparency = true,
    shading = true
)

# CairoMakie.xlims!(ax, -1.3, 1.3)
# CairoMakie.ylims!(ax, -1.3, 1.3)
# CairoMakie.zlims!(ax, -1.3, 1.3)

ax.azimuth[]   =  π/20
ax.elevation[] =  π/10
CairoMakie.hidespines!(ax)
CairoMakie.axislegend(ax; 
    position = :rt,
    labelsize = 36,        # Default is usually ~12-16. Increase this for text.
    patchsize = (100, 80),  # (width, height). Makes the line sample longer.
    linewidth = 3,         # Makes the line inside the legend thicker.
    padding = (10, 10, 10, 10) # Adds space inside the legend box.
)

CairoMakie.hidexdecorations!(ax, grid = true)
CairoMakie.hideydecorations!(ax, grid = true)
CairoMakie.hidezdecorations!(ax, grid = true)

f



UndefVarError: UndefVarError: `prob` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [16]:
save("./artifacts/plots/bloch_def.svg", f, px_per_unit = 2)

UndefVarError: UndefVarError: `f` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [17]:
var_fid = unitary_rollout_fidelity(var_prob.trajectory, sys)
tog_fid = unitary_rollout_fidelity(tog_prob.trajectory, sys)
def_fid = unitary_rollout_fidelity(prob.trajectory, sys)


UndefVarError: UndefVarError: `prob` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [18]:
fig = Figure()
ax = Axis(
    fig[1, 1], 
    title="Fidelity Comparison",
    ylabel="Fidelity",
    xlabel="Method",
    xticks=(1:3, ["Variational", "Toggle", "Default"])
)

tog_obj()
var_obj()


# Create bar chart with the three fidelity values
barplot!(
    ax,
    1:3,
    [var_fid, tog_fid, def_fid],
    color=[:steelblue, :coral, :seagreen],
    strokewidth=1,
    strokecolor=:black
)

fig

MethodError: MethodError: no method matching tog_obj()
The function `tog_obj` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  tog_obj(!Matched::NamedTrajectory, !Matched::Vector{Matrix{ComplexF64}}, !Matched::Matrix{ComplexF64})
   @ Main ~/Documents/research/pulses/project/notebooks/src/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X10sZmlsZQ==.jl:1


In [19]:
"""
    upsample_constant(vals, dts; factor=2)

Take control values `vals` with time steps `dts` (same length),
and upsample by `factor`, returning (vals_up, dts_up).
"""
function upsample_constant_controls(vals::AbstractArray; factor::Int=2)
    vals_up = repeat(vals, inner=factor)
    return vals_up
end

# Example
vals = [1, 2, 3]
dts  = [0.2, 0.2, 0.2]

vals_up = upsample_constant_controls(vals; factor=3)

println(vals_up)  # [1, 1, 2, 2, 3, 3]
#println(dts_up) 

function upsample_matrix(controls::AbstractArray, dts::AbstractArray; factor::Int=2)
    new_controls = []
    for c in eachrow(controls)
        new_c = upsample_constant_controls(c; factor=factor)
        T = length(c)
        push!(new_controls, new_c)
    end
    dts_up = dts[1] / factor .* ones(length(dts) * factor*T)
    new_controls = reduce(vcat, [v' for v in new_controls])
    return new_controls, dts_up
end
function tog_obj_upsample(
    traj::NamedTrajectory, 
    H_drives::Vector{Matrix{ComplexF64}},
    H_error::Matrix{ComplexF64};
    factor::Int=1
)
    T = traj.T * factor
    controls = traj.a
    a_new, Δt_new = upsample_matrix(traj.a, traj.Δt; factor=factor)

    sys = QuantumSystem(H_drives)
    U = iso_vec_to_operator.(eachcol(unitary_rollout(a_new, Δt_new, sys)))
    
    # Toggle integral (truncate at (traj.T - 1) * factor)
    H_ti = sum(Δt_new[i] .* U[i]' * H_error * U[i] for i = 1:(traj.T - 1) * factor)

    d₁ = size(U[1], 1)
    Δt₁ = Δt_new[1]
    metric = norm(tr(H_ti'H_ti)) / (T * Δt₁)^2 / d₁
    return metric
end


[1, 1, 1, 2, 2, 2, 3, 3, 3]


tog_obj_upsample (generic function with 1 method)